# Tests del MEGC Stargate

Carga el modelo desde `stargate_cge.ipynb` y verifica que:

1. Sin datacenter, el equilibrio replica la calibración (precios base).
2. Con el shock, $p_E$ sube, $E_D = 4{,}55$ TWh y se cumple la ley de Walras.

Cada test se corre en ambos cierres (corto y largo plazo). Si alguno falla, la última celda lanza `AssertionError`.

In [1]:
# Carga del modelo: stargate_cge.ipynb tiene que estar en la misma carpeta que este notebook.
# En Google Colab cada notebook se abre solo (en /content), así que si falta se pide subirlo.
from pathlib import Path

MODELO = Path("stargate_cge.ipynb")
if not MODELO.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            f"No se encontró {MODELO.resolve()}. Abrí este notebook desde la carpeta que contiene stargate_cge.ipynb."
        ) from None
    print("No se encontró stargate_cge.ipynb. Subilo con el botón de abajo:")
    subidos = [n for n in files.upload() if n.endswith(".ipynb")]
    if not subidos:
        raise FileNotFoundError("No se subió ningún .ipynb con el modelo.")
    Path(subidos[0]).rename(MODELO)  # Colab renombra duplicados como "stargate_cge (1).ipynb"

%run stargate_cge.ipynb

In [2]:
import math

cal = calibrar(EconomiaBase(), ShockDatacenter())
CIERRES = ["corto", "largo"]


def approx(a, b, abs_tol=1e-6):
    return math.isclose(a, b, rel_tol=1e-6, abs_tol=abs_tol)

## Test 1: el escenario base replica la calibración

In [3]:
def test_base_replica_calibracion(cierre):
    b = resolver(cal, cierre, con_dc=False)
    assert approx(b["pE"], cal["pE0"])
    assert approx(b["w"], 1.0)
    assert approx(b["rE"], 1.0)
    assert approx(b["XE"], cal["XE0"])

## Test 2: el shock sube $p_E$ y cumple Walras

In [4]:
def test_shock_sube_precio_energia_y_cumple_walras(cierre):
    b = resolver(cal, cierre, con_dc=True)
    assert b["pE"] > cal["pE0"]
    assert approx(b["ED"], 4.55)
    absorcion = b["pE"] * b["ED"] + b["w"] * b["LD"] + b["Tax"]
    assert approx(b["CR"] - b["XR"], absorcion, abs_tol=1e-8)

## Ejecución

In [5]:
fallas = []
for test in [test_base_replica_calibracion, test_shock_sube_precio_energia_y_cumple_walras]:
    for cierre in CIERRES:
        nombre = f"{test.__name__}[{cierre}]"
        try:
            test(cierre)
            print(f"PASS  {nombre}")
        except AssertionError as e:
            fallas.append(nombre)
            print(f"FAIL  {nombre}  {e}")

assert not fallas, f"{len(fallas)} test(s) fallaron: {fallas}"
print(f"\nTodos los tests pasaron ({2 * len(CIERRES)}).")

PASS  test_base_replica_calibracion[corto]
PASS  test_base_replica_calibracion[largo]
PASS  test_shock_sube_precio_energia_y_cumple_walras[corto]
PASS  test_shock_sube_precio_energia_y_cumple_walras[largo]

Todos los tests pasaron (4).
